In [258]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
from numpy import random
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode,GeographicPolar, Geographic
from glob import glob
import matplotlib.pyplot as plt

from parcels import ScipyParticle, AdvectionRK4,AdvectionRK4_3D

In [259]:
sys.path.append(os.getcwd()+'/Source_vvl')

from OP_Kernels import *
from OP_functions import *

In [260]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')

def finder(jjii,lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

finder(jjii,49.35844243668596,-123.80204117521916)

(512, 251)

In [261]:
def load_config(config_yaml):
   with open(config_yaml) as f:
       config = yaml.safe_load(f)
   return config

In [262]:
def pandas_deploy(N,MFc,r,dd,dtp):
    n =1
    MFc = float(MFc)
    Outfall_deploy = pd.read_csv(N, index_col = [0])
    Pol = list(Outfall_deploy.Population)
    Lat = Outfall_deploy.Lat
    Lon = Outfall_deploy.Lon
    Depth = Outfall_deploy.Depth
    clat = []
    clon = []
    cz = []
    for i,loc in enumerate(Pol):
        for j in range(int(round((loc*250*dtp)/MFc,0))):
            clat.append(Lat.iat[i])
            clon.append(Lon.iat[i])
            try:
                cz.append(float(Depth.iat[i]))
            except ValueError: #This is for rivers
                cz.append(2)
    N = len(clat)
    deg2m = 111319.5 * np.cos(49 * np.pi / 180)
    var = (r / (deg2m * 3))**2
    x_offset, y_offset = np.random.multivariate_normal([0, 0], [[var, 0], [0, var]], [n,N]).T #Here we define a cloud of deployment points.
    z_offset=np.random.random_sample([N]).T*(dd) #add some depth variability
    lon = np.zeros([N])
    lat = np.zeros([N])
    z = np.zeros([N])
    for i in range(N):
        lon[i]=(clon[i] + x_offset[i])
        lat[i]=(clat[i] + y_offset[i])
        z[i]=(cz[i] + z_offset[i])
    return lon,lat,z

In [263]:
coord = xr.open_dataset(path('salish')['coords'],decode_times=False)
mask = xr.open_dataset(path('salish')['mask'],decode_times=False)

In [264]:
def p_unidist(lat0,lon0,bat,dy,dx):
    latbat = np.zeros(bat.shape)
    lonbat = np.zeros(bat.shape)

    for i in range(bat.shape[0]):
        for j in range(bat.shape[1]):
            if bat[i,j]>1e-5:
                latbat[i,j] = lat0[i,j]
                lonbat[i,j] = lon0[i,j]

    yi = np.arange(0,latbat.shape[0],dy)
    xi = np.arange(0,latbat.shape[1],dx)
    plat0,plon0 = latbat[yi,:],lonbat[yi,:]
    plat1,plon1 = plat0[:,xi],plon0[:,xi]

    plon,plat = [],[]
    for i in range(plat1.shape[0]):
        for j in range(plat1.shape[1]):
            if plat1[i,j]>1e-5:
                plat.append(plat1[i,j])
                plon.append(plon1[i,j])
    return plat,plon

In [265]:
#clat,clon = p_unidist(coord.nav_lat,coord.nav_lon,mask.totaldepth,20,20)
# np.save('lat.npy',np.array(clat))
# np.save('lon.npy',np.array(clon))

In [266]:
clat = np.load('lat.npy')
clon = np.load('lon.npy')

In [267]:
cz = np.ones(len(clat))

In [268]:
param = load_config('yaml/config.yaml')
start = datetime(param['startdate']['year'], param['startdate']['month'], param['startdate']['day']) #Start date
length = 1#param['param']['length'] # Set Time length [days] 
dt = param['param']['dt'] #toggle between - or + to pick backwards or forwards 
N = param['param']['N'] # Name of deploy loc file
dd = param['param']['dd'] #max depth difference z in N
name = param['file']['name'] #name output file
dtp = param['param']['dtp'] #how often particle released in hours
odt = param['param']['odt'] #how often data is recorded
rrr = param['param']['r'] #radious of particle deployment
SI = param['param']['SI'] #how many particles per super-individual

length = 1
duration = timedelta(days=length)
print(f"The model will run for {duration.days} days, starting at {start}")

n = 1 
N = len(clat)
deg2m = 111319.5 * np.cos(49 * np.pi / 180)
var = (rrr / (deg2m * 3))**2
x_offset, y_offset = np.random.multivariate_normal([0, 0], [[var, 0], [0, var]], [n,N]).T #Here we define a cloud of deployment points.
z_offset=np.random.random_sample([N]).T*(dd) #add some depth variability
lon = np.zeros([N])
lat = np.zeros([N])
z = np.zeros([N])
for i in range(N):
    lon[i]=(clon[i] + x_offset[i])
    lat[i]=(clat[i] + y_offset[i])
    z[i]=(cz[i] + z_offset[i])


print(f"The model will release {N} particle every timestep")

daterange = [start+timedelta(days=i) for i in range(length)]
fn =  name + '.zarr'
outfile = os.path.join(os.getcwd()+'/results/', fn)

print(f"Output file: {outfile}")

The model will run for 1 days, starting at 2023-01-02 00:00:00
The model will release 217 particle every timestep
Output file: /ocean/jvalenti/MOAD/analysis-jose/notebooks/MP_model/results/test_kernels.zarr


/tmp/ipykernel_770967/492534788.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lon[i]=(clon[i] + x_offset[i])
/tmp/ipykernel_770967/492534788.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lat[i]=(clat[i] + y_offset[i])


In [269]:
#Define our Parcels Particles
class MPParticle(JITParticle):    
    fact = Variable('fact', initial = 1)
    status = Variable('status', initial = 1) #0=non initialized, 1=active, 2=beached, 3 = sedimented, 5=export to PO, 10=lost
    dim1 = Variable('dim1', initial = 7.5e-6) # particle small semiaxis
    dim2= Variable('dim2', initial = 7.5e-6)  # particle medium semiaxis
    dim3 = Variable('dim3', initial = 2e-4) # particle large semiaxis
    shape = Variable('shape', initial = 1.0) #particle roundness (0 is cube, 1 is sphere)
    density = Variable('density', initial = 1350)  #particle density 

In [270]:
"""
Su, Fan (2022)

Particle shape:
    a >= b >= c  : semi-axes of the particle
    e (epsilon)  : (e = 1 ellipsoid, e -> 0 box-like)

0.2 <= e <= 1.8 (CSF 0.09-1, sphericity 0.31-1).
"""

def terminal_velocity(a, b, c, rho_p, rho_f, mu=1.002e-3, e=1.0):
    """
    Estimate terminal sinking velocity (m/s) of a super-ellipsoidal particle.

    Parameters
    ----------
    a, b, c : particle dimensions in metres (semiaxis). 
    rho_p   : particle density (kg/m^3)
    rho_f   : fluid density (kg/m^3)
    mu      : dynamic viscosity of fluid (Pa.s); default water at ~20 C
    e       : shape parameter epsilon (1 = ellipsoid, 0.2-1.8 range tested)
    """
    t = 14
    mu = 4.2844e-5 + 1/(0.157*((t + 64.993)**2)-91.296) #kinematic viscosity for Temp of SSC
    g = 9.81 
    fact = 1
    if rho_p <= rho_f:
        fact = -1 #raise ValueError
    a, b, c = sorted((a, b, c), reverse=True)
    
    EI = b / a

    if EI <= 0.1:
        #fiber path through Komar
        print('fiber')
        l = a*2
        d = b + c
        u_t = ((l/d)**-1.664)*0.079*((l**2)*g*(abs(rho_p - rho_f)))/(mu)
    else:
        print('fragment')
        #fragment path through Fan
        FI = c / b
        # equivalent-volume sphere diameter
        V = 2.0*a*b*c * (e**2) * math.gamma(e/2 + 1)*math.gamma(e)/math.gamma(3*e/2 + 1) * math.gamma(e/2)*math.gamma(e/2)/math.gamma(e)
        d_eq = (6.0 * V / math.pi) ** (1.0 / 3.0)

        dr = abs(rho_p - rho_f)
        d_star = d_eq * (rho_f * dr * g / mu ** 2) ** (1.0 / 3.0)

        # sphere reference and shape correction
        u_star_S = 1.0 / (18.0 / d_star ** 2 + 2.412 / (4.0 * math.sqrt(d_star)))
        
        L = math.log10(d_star) #(Eq. 31b)
        C1 = 0.11 * L + 0.17
        C2 = 0.21 * L + 0.17
        C3 = 0.48 * L - 0.56
        C4 = 1.05 * L - 1.38
        
        u_star = EI ** C1 * FI ** C2 * (C3 * e ** 2 + C4 * e + 1.0 - C3 - C4) * u_star_S
        u_t = u_star * (dr * mu * g / rho_f ** 2) ** (1.0 / 3.0)

    return fact*u_t

# Example: sphere, 10 mm diameter, resin (1200 kg/m3) in water
r = terminal_velocity(7.5e-6, 7.5e-6, 200e-6, 1350, 1000, mu=1.002e-3, e=1)
print(f"{round(r, 10)} m/s")


fiber
0.0001574134 m/s


In [272]:
cs = {
        "time": ("time", 1),
        "depth": ("depth", 40),   
        "lat": ("latitude", 100),
        "lon": ("longitude", 100),
    }

In [273]:
def get_Winds_path(date):
    """Construct Winds results path given the date
    :rtype: str
    """
    datestr = date.strftime('y%Ym%md%d')
    # Make winds path
    if date > datetime(2023,2,22):
        path = '/results/forcing/atmospheric/continental2.5/nemo_forcing'
        path = os.path.join(path, f'hrdps_{datestr}.nc')
    else:
        path = '/results/forcing/atmospheric/GEM2.5/operational'
        path = os.path.join(path, f'ops_{datestr}.nc')
    return path  

In [274]:
def get_Biol_path(date):
    # Make biol path
    path = '/data/jvalenti/results/'
    datestr = [date.strftime(fmt) for fmt in ('%d%b%y', '%Y%m%d-%Y%m%d')]
    path = os.path.join(path, datestr[0].lower(), f'SHEM_1h_{datestr[1]}_biol_T.nc')
    return path  

In [276]:
duration = timedelta(days=length)
paths = path('salish')
cgrid_dim = {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'}
Ulist, Vlist, Wlist, Tlist, Blist, Waveslist, Windlist = [], [], [], [], [], [],[]

for day in range(duration.days):
  path_NEMO = make_prefix(start + timedelta(days=day),paths['NEMO'])
  Ulist.append(path_NEMO + '_grid_U.nc')
  Vlist.append(path_NEMO + '_grid_V.nc')
  Wlist.append(path_NEMO + '_grid_W.nc')
  Tlist.append(path_NEMO + '_grid_T.nc')

  Blist.append(get_Biol_path(start + timedelta(days=day)))
  Waveslist.append(get_WW3_path(start + timedelta(days=day)))
  Windlist.append(get_Winds_path(start + timedelta(days=day)))


filenames = {
  'U': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Ulist},
  'V': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Vlist},
  'W': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Wlist}
}
dimensions = {
  'U': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'V': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'W': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
}
variables = {
  'U': 'vozocrtx',
  'V': 'vomecrty',
  'W': 'vovecrtz'
}

field_set = FieldSet.from_nemo(
    filenames,
    variables,
    dimensions,
    allow_time_extrapolation=True,
    chunksize=cs
)

Kz = Field.from_netcdf(filenames['W'], 'vert_eddy_diff', dimensions['W'],allow_time_extrapolation=True, chunksize=cs)
field_set.add_field(Kz)

filenames['tracer'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Tlist[0],'data': Tlist}
dimensions['tracer'] = {'lon': 'glamt', 'lat': 'gphit', 'depth': 'deptht','time': 'time_counter'}

R = Field.from_netcdf(filenames['tracer'], 'sigma_theta', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
T = Field.from_netcdf(filenames['tracer'], 'votemper', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)

field_set.add_field(R)
field_set.add_field(T)

#BIOLOGY fields
filenames['Biology'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Blist[0],'data': Blist}

#####FROM HERE ALL NOT CHUNKED FIELDS (2D or different dimensions)
#Add Stokes Drift fields
dimensions['stokes'] = {'lon': 'longitude', 'lat': 'latitude', 'time': 'time'}
filenames['stokes'] = {'lon': paths['coordsWW3'], 'lat': paths['coordsWW3'], 'data': Waveslist}
us = Field.from_netcdf(filenames['stokes'],'uuss', dimensions['stokes'],allow_time_extrapolation=True)
vs = Field.from_netcdf(filenames['stokes'], 'vuss', dimensions['stokes'],allow_time_extrapolation=True)
wl = Field.from_netcdf(filenames['stokes'], 'lm', dimensions['stokes'],allow_time_extrapolation=True)
us.units = GeographicPolar()   # u: m/s -> deg lon/s 
vs.units = Geographic() 
field_set.add_field(us)
field_set.add_field(vs)
field_set.add_field(wl)
field_set.add_vector_field(VectorField("stokes", us, vs, wl))

filenames['totaldepth'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': paths['mask']}
filenames['ssh'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': Tlist}
variables['totaldepth']= 'totaldepth'
variables['ssh']= 'sossheig'

#Add Bathymetry 2D field
dimensions['flat'] = {'lon': 'glamt', 'lat': 'gphit'}
TD = Field.from_netcdf(filenames['totaldepth'], variables['totaldepth'], dimensions['flat'],allow_time_extrapolation=True)
field_set.add_field(TD)

#Add SSH 
dimensions['flat_t'] = {'lon': 'glamt', 'lat': 'gphit','time': 'time_counter'}
SSH = Field.from_netcdf(filenames['ssh'], variables['ssh'], dimensions['flat_t'],allow_time_extrapolation=True)
field_set.add_field(SSH)

b_stress = Field.from_netcdf(filenames['Biology'], 'bottom_stress', dimensions['flat_t'],allow_time_extrapolation=True)
field_set.add_field(b_stress)

filenames['wind'] = {'lon': Windlist[0],'lat': Windlist[0],'data': Windlist}
dimensions['wind'] = {'lon': 'nav_lon', 'lat': 'nav_lat','time': 'time_counter'}

u_wind = Field.from_netcdf(filenames['wind'], 'u_wind', dimensions['wind'],allow_time_extrapolation=True)
v_wind = Field.from_netcdf(filenames['wind'], 'v_wind', dimensions['wind'],allow_time_extrapolation=True)
u_wind.units = GeographicPolar()   # u: m/s -> deg lon/s 
v_wind.units = Geographic()  
field_set.add_field(u_wind)
field_set.add_field(v_wind)
field_set.add_vector_field(VectorField("wind", u_wind, v_wind))

/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might be wrongly parsed.
  with _grid_fb_class(
/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/field.py:648: FileWarning: File /ocean/jvalenti/MOAD/grid/coordinates_seagrid_SalishSea201702.nc could not be decoded properly by xarray (version 2024.6.0). It will be opened with no decoding. Filling values might b

## Define constants

In [277]:
field_set.add_constant('deg2met', 111319.5)
#field_set.add_constant('latT', 0.6495)   #Average value for SoG #cos(particle.lat*(math.pi/180))
field_set.add_constant('deglatT', 72302)   #deglatT == latT*deg2met
field_set.add_constant('kh', 1.5)  
field_set.add_constant('SDD', 3.34e-6) #std Diameter
field_set.add_constant('SDL', 24.4e-5) #std length

Ub = param['param']['Ub']
PrUb = 1- np.exp(-dt/Ub)
field_set.add_constant('PrUb', PrUb) #Ub=timescale unbeaching in seconds days * 86400  Pr = 1 - exp(-particle.dt/Ub)

## Define Sources

In [278]:
lat = np.ones(len(clat))*49.33451996322204
lon = np.ones(len(clat))*-123.83734250024288
z = np.ones(len(clat))*0.01

In [279]:
pset = ParticleSet.from_list(field_set, MPParticle, lon=lon, lat=lat, depth=z, lonlatdepth_dtype=np.float32)

In [280]:
def Advection(particle, fieldset, time): 
    if particle.status == 1: #If particle is not active, do not move it
        ssh = fieldset.sossheig[time, particle.depth, particle.lat, particle.lon,particle] #SSH(t)
        sshn = fieldset.sossheig[time+particle.dt, particle.depth, particle.lat, particle.lon,particle] #SSH(t+dt)
        td = fieldset.totaldepth[time, particle.depth, particle.lat, particle.lon,particle]#Total_depth
        particle.fact = (1+ssh/td) #VVl conversion factor
        VVL = (sshn-ssh)*particle.depth/td #VVl w velocity correction
        #4th order RK
        (u1, v1, w1) = fieldset.UVW[particle]
        lon1 = particle.lon + u1*.5*particle.dt
        lat1 = particle.lat + v1*.5*particle.dt
        dep1 = particle.depth + w1*.5*particle.dt/particle.fact
        (u2, v2, w2) = fieldset.UVW[time + .5 * particle.dt, dep1, lat1, lon1,particle]
        lon2 = particle.lon + u2*.5*particle.dt
        lat2 = particle.lat + v2*.5*particle.dt
        dep2 = particle.depth + w2*.5*particle.dt/particle.fact
        (u3, v3, w3) = fieldset.UVW[time + .5 * particle.dt, dep2, lat2, lon2,particle]
        lon3 = particle.lon + u3*particle.dt
        lat3 = particle.lat + v3*particle.dt
        dep3 = particle.depth + w3*particle.dt/particle.fact
        (u4, v4, w4) = fieldset.UVW[time + particle.dt, dep3, lat3, lon3,particle]
        particle_dlon += (u1 + 2*u2 + 2*u3 + u4) / 6. * particle.dt
        particle_dlat += (v1 + 2*v2 + 2*v3 + v4) / 6. * particle.dt
        particle_ddepth += (w1 + 2*w2 + 2*w3 + w4) /6. * particle.dt/particle.fact + VVL
        if particle_ddepth + particle.depth < 0:
            particle_ddepth = -(2*particle.depth + particle_ddepth) #reflection on surface

In [281]:
def vert_mix(particle,fieldset,time):
    """Vertical mixing"""
    if particle.status==1:
        #Vertical mixing
        if particle.depth + 0.5/particle.fact > td: #Only calculate gradient of diffusion for particles deeper than 0.5 otherwise OP will look for particles outside the domain raising an error.
            Kzdz = 2*(fieldset.vert_eddy_diff[time, particle.depth, particle.lat, particle.lon,particle]-fieldset.vert_eddy_diff[time, particle.depth-0.5/particle.fact, particle.lat, particle.lon]) 
            #backwards difference 
        else: 
            Kzdz = 2*(fieldset.vert_eddy_diff[time, particle.depth+0.5/particle.fact, particle.lat, particle.lon]-fieldset.vert_eddy_diff[time, particle.depth, particle.lat, particle.lon,particle]) #forward difference 
        dgrad = Kzdz*particle.dt/particle.fact
        if particle.depth+(0.5*dgrad) > 0 and particle.depth+(0.5*dgrad) < td:
            Kz = fieldset.vert_eddy_diff[time, particle.depth+ 0.5*dgrad, particle.lat, particle.lon] 
            #Vertical diffusivity SSC  
        else:
            Kz = 0 #Otherwise it would check outside the domain

        Rr = ParcelsRandom.uniform(-1, 1)
        d_random = sqrt(3*2*Kz*particle.dt) * Rr/particle.fact
        dzs = (dgrad + d_random)
        
        #Apply turbulent mixing.
        if dzs + particle_ddepth + particle.depth > td: #crossed bottom in dt (Reflect)
            particle_ddepth = 2*td - (dzs + particle_ddepth + 2*particle.depth) #bounce on boundary/ no energy loss
        elif dzs + particle.depth + particle_ddepth < 0:#crossed surface in dt (#Well mixed boundary layer)
            particle_ddepth = -(dzs + 2*particle.depth+ particle_ddepth) 
        else:
            particle_ddepth += dzs #apply mixing  

In [282]:
def horz_mix(particle,fieldset,time):
    """Horizontal mixing (Beaching)"""
    if particle.status==1:
        Rrx = ParcelsRandom.uniform(-1, 1)
        Rry = ParcelsRandom.uniform(-1, 1)
        d_x = (sqrt(3*2*fieldset.kh*particle.dt) * Rrx)/fieldset.deglatT
        d_y = (sqrt(3*2*fieldset.kh*particle.dt) * Rry)/fieldset.deg2met   
        d_randomx = particle.lon + d_x + particle_dlon
        d_randomy = particle.lat + d_y + particle_dlat
        Sbh = 1
        Swh = 1
        if particle.depth < 5:
            Sbh = fieldset.sigma_theta[time, 0.51, d_randomy, d_randomx] #Check if particles reach coast at shallow water (SW Density = 0)
            if Sbh==0:
                particle.status = 2 #beached
        else:
            Swh = fieldset.sigma_theta[time, particle.depth, d_randomy, d_randomx]
            if Swh == 0:
                #Do not cross wall Keep particle in place
                particle.status = -1
                
        #Apply horizontal mixing (beaching for particles pushed through coast) 
        if particle.status == 1:
            particle_dlat += d_y
            particle_dlon += d_x
        elif particle.status == -1:
            particle.status = 1 #Do nothing, and resuspend. This particles crossed bathymetry at deep water

In [283]:
def Windage(particle, fieldset, time):
    if particle.status == 1 and particle.depth < 0.1:
        (uwind, vwind) = fieldset.wind[time, particle.depth, particle.lat, particle.lon+360,particle]
        particle_dlon += uwind * 0.03 * particle.dt 
        particle_dlat += vwind * 0.03 * particle.dt

In [284]:
def Stokes_drift(particle, fieldset, time):
    """Apply Stokes drift calculated by WW3"""
    if particle.status == 1 and particle.depth < 5:

        if particle.lat > 48 and particle.lat < 51: #particle is inside WW3 data field
            (us0, vs0, wl) = fieldset.stokes[time, 0, particle.lat, particle.lon,particle]
            k = (2*math.pi)/wl
            us = (us0*exp(-math.fabs(2*k*particle.depth)))
            vs = (vs0*exp(-math.fabs(2*k*particle.depth)))
        else:
            us = 0
            vs = 0
        particle_dlon += us * particle.dt
        particle_dlat += vs * particle.dt

In [285]:
def Buoyancy(particle, fieldset, time):
    """Calculating settling velocity using Komar cylinder or Fan super ellipsoids W_s"""
    if particle.status == 1: #Check particle is in the water column   
        deps = max(particle.depth,0.51) #Surface value of Tracer
        c = particle.dim1 # particle small semiaxis
        b = particle.dim2 # particle medium semiaxis
        a = particle.dim3 # particle large semiaxis
        e = particle.shape #particle roundness (0 is cube, 1 is sphere)
        rho_p = particle.density #particle density 

        g = 9.81 #gravity acceleration
        t = fieldset.votemper[time, deps, particle.lat, particle.lon] #Loading temperature from SSC
        rho_f = fieldset.sigma_theta[time, deps, particle.lat, particle.lon] + 1000 #Loading density sw from SSC
        mu = 4.2844e-5 + 1/(0.157*((t + 64.993)**2)-91.296) #kinematic viscosity for Temp of SSC
        dr = math.fabs(rho_p-rho_f)

        #IS THE PARTICLE SINKING OR FLOATING
        sinking = 1
        if rho_p <= rho_f:
            sinking = -1 #raise ValueError

        EI = b / a 

        if EI <= 0.1: #CHECK if it's a fiber
                #fiber path through Komar
                l = a*2
                d = b + c
                W_s = ((l/d)**-1.664)*0.079*((l**2)*g*(dr))/(mu)
        else:     
            #fragment path through Fan
            FI = c / b
            # equivalent-volume sphere diameter
            Vol = 2.0*a*b*c * (e**2) * math.gamma(e/2 + 1)*math.gamma(e)/math.gamma(3*e/2 + 1) * math.gamma(e/2)*math.gamma(e/2)/math.gamma(e)
            d_eq = (6.0 * Vol / math.pi) ** (1.0 / 3.0)
    
            d_star = d_eq * (rho_f * dr * g / mu ** 2) ** (1.0 / 3.0)
    
            # sphere reference and shape correction
            u_star_S = 1.0 / (18.0 / d_star ** 2 + 2.412 / (4.0 * math.sqrt(d_star)))
            
            L = math.log10(d_star) #(Eq. 31b)
            C1 = 0.11 * L + 0.17
            C2 = 0.21 * L + 0.17
            C3 = 0.48 * L - 0.56
            C4 = 1.05 * L - 1.38
            
            u_star = EI ** C1 * FI ** C2 * (C3 * e ** 2 + C4 * e + 1.0 - C3 - C4) * u_star_S
            W_s = u_star * (dr * mu * g / rho_f ** 2) ** (1.0 / 3.0)
        
        dws = sinking * W_s*particle.dt/particle.fact
        
        #Apply buoyancy
        if particle.depth + dws + particle_ddepth > td:
            particle_ddepth = td-particle.depth #particle reaches the bottom 
            vtau_crit = (0.30 / (1.0 + 1.2 * d_star) + 0.055 * (1.0 - exp(-0.020 * d_star))) * g * d_eq
            vtau = fieldset.bottom_stress[time,particle.depth, particle.lat, particle.lon,particle]
            if vtau <= vtau_crit: #Assess critical shear here to avoid sedimentation
                particle.status = 3 #Trap particle in sediment

        elif particle.depth + dws + particle_ddepth < 0: #particle reaches the surface 
            particle_ddepth = - particle.depth 
        else:
            particle_ddepth += dws

In [286]:
def Unbeaching(particle, fieldset, time):
    '''Resuspension prob'''  
    if particle.status == 2:      
        if ParcelsRandom.uniform(0,1)<fieldset.PrUb:
            particle.status = 1

def resuspension(particle, fieldset, time):
    # proceed if particle is at the seabed
    if particle.status == 3:
        vtau_crit = fieldset.tau_crit #Value to check
        vtau = fieldset.Bot_stress[time,particle.depth, particle.lat, particle.lon,particle]
        if vtau <= vtau_crit:
            particle.status = 1
            #particle_ddepth += #HOW MUCH should we resuspend?

In [287]:
def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        if Sbh == 0: #Ready to beach particles
            print('particle beached')
            particle.state = StatusCode.Success
            #particle.status = 2 #beached DO NOT MOVE outside domain
        elif Swh ==0:
            print('particle hit wall') #Do not cross wall Keep particle in place
            particle.state = StatusCode.Success
        else:
            print('lost Particle')
            particle.status = 10 #lost
            particle.delete()
        
def KeepInOcean(particle, fieldset, time):
    if particle.state == StatusCode.ErrorThroughSurface:
        particle.depth = 0.0
        particle.state = StatusCode.Success

def export(particle,fieldset,time):
    if particle.status==1:
        Pacific =  -particle.lat*0.84120957 -83.98027258 #Checking if particle gets too close to boundary JdF
        South = particle.lon*0.35157547 +90.26497859 #Checking southern boundary model
        if particle.lon<Pacific:
            particle.status = 5 #Exported to Pacific Ocean
        if particle.lat<South:
            if particle.depth < 5:
                particle.status =2 #Beached
            else:
                particle_dlat += 0.1*particle.dt/fieldset.deg2met 
                #Northward velocity to avoid particles from getting stuck on the wall only an issue at southernmost cell of SSC

In [288]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

In [289]:
pset.execute(
    [Advection,export,KeepInOcean,CheckOutOfBounds,horz_mix,vert_mix,Stokes_drift,Windage,Buoyancy],
    runtime=timedelta(hours=12),
    dt=timedelta(seconds=dt),
    output_file=pset.ParticleFile(name='/home/jvalenti/MOAD/results/output.zarr', outputdt=timedelta(minutes=30)))

INFO: Output files are stored in /home/jvalenti/MOAD/results/output.zarr.
100%|██████████| 43200.0/43200.0 [00:56<00:00, 763.86it/s]
